<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-5-agents/Module_5_Session_2_ReAct_Agent_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 — Session 2: The ReAct Agent Loop

## What are we building?
A Swiggy support agent that thinks in a loop — Reason, Act, Observe,
repeat — until it has enough information to give a final answer.
This is the ReAct pattern that powers every agent framework in the world.

## Why does this matter?
Session 1 was one tool call and done. Real problems need multiple steps.
ReAct is how agents handle multi-step reasoning without human help.

## What we will learn:
- What the ReAct loop looks like in raw Python
- How conversation history grows with each tool call
- How the LLM decides when it has enough information to stop
- Why frameworks like LangGraph exist (to manage this loop for you)

## Model: openai/gpt-oss-20b on Groq
## Platform: Google Colab
## API: Groq (same API key as always)

## Step 1: Install and Setup
Same setup as Session 1. No new libraries.

In [ ]:
# Same setup as Session 1 — no new libraries needed
!pip install groq -q

from groq import Groq
import os
import json

from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq()
MODEL = "openai/gpt-oss-20b"

print("Setup complete. Model:", MODEL)

## Step 2: Define Tools
Same two tools as Session 1 — check order status and calculate refund.
We also add a third tool: escalate_to_human.
This is how a real agent knows when to give up and hand off to a human agent.

In [ ]:
# --- TOOL 1: Check Swiggy Order Status ---
def check_order_status(order_id):
    orders = {
        "ORD001": {"status": "Delayed", "eta": "45 minutes", "restaurant": "Burger King"},
        "ORD002": {"status": "Delivered", "eta": None, "restaurant": "Pizza Hut"},
        "ORD003": {"status": "Preparing", "eta": "25 minutes", "restaurant": "McDonald's"},
    }
    order = orders.get(order_id)
    if order:
        return f"Order {order_id} from {order['restaurant']}: {order['status']}. ETA: {order['eta']}"
    else:
        return f"Order {order_id} not found in system."

# --- TOOL 2: Calculate Refund ---
def calculate_refund(order_id, reason):
    order_values = {"ORD001": 450, "ORD002": 320, "ORD003": 180}
    amount = order_values.get(order_id, 0)
    if reason == "wrong_item":
        refund = amount * 1.0
    elif reason == "late_delivery":
        refund = amount * 0.3
    else:
        refund = 0
    return f"Refund for {order_id}: ₹{refund} approved for reason: {reason}"

# --- TOOL 3: Escalate to Human --- NEW
def escalate_to_human(order_id, reason):
    # When agent cannot resolve — hand off to human support
    return f"Order {order_id} escalated to human support team. Reason: {reason}. Ticket created."

# Tool dispatcher — maps name to function
available_tools = {
    "check_order_status": check_order_status,
    "calculate_refund": calculate_refund,
    "escalate_to_human": escalate_to_human
}

# Tool descriptions for LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_order_status",
            "description": "Check the current status of a Swiggy order. Use this first before deciding on refunds.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The Swiggy order ID"}
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_refund",
            "description": "Calculate refund for a Swiggy order. Use this only after checking order status.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The Swiggy order ID"},
                    "reason": {
                        "type": "string",
                        "description": "Reason for refund — wrong_item or late_delivery",
                        "enum": ["wrong_item", "late_delivery"]
                    }
                },
                "required": ["order_id", "reason"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_human",
            "description": "Escalate the issue to a human support agent when you cannot resolve it yourself.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The Swiggy order ID"},
                    "reason": {"type": "string", "description": "Why this is being escalated"}
                },
                "required": ["order_id", "reason"]
            }
        }
    }
]

print("Tools defined:", len(tools))
for t in tools:
    print(" -", t["function"]["name"])

In [ ]:
def run_react_agent(customer_query, max_steps=5):
    """
    ReAct loop:
    - Reason: LLM decides what to do next
    - Act: We run the tool
    - Observe: We add result to conversation history
    - Repeat until LLM gives final answer or max_steps reached
    """

    print(f"Customer: {customer_query}")
    print("=" * 60)

    # Conversation history grows with every step
    # This is how the LLM remembers what it has already done
    messages = [
        {"role": "system", "content": """You are a helpful Swiggy customer support agent.
        When a customer has a complaint, first check the order status, then take action.
        Only give a final answer when you have all the information you need."""},
        {"role": "user", "content": customer_query}
    ]

    # --- THE REACT LOOP ---
    for step in range(max_steps):
        print(f"Step {step + 1}:")

        # REASON: Ask LLM what to do next
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        # ACT: Did LLM decide to call a tool?
        if message.tool_calls:
            tool_call = message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"  Reason: LLM wants to call '{tool_name}'")
            print(f"  Act: Running {tool_name}({tool_args})")

            # Run the tool
            tool_result = available_tools[tool_name](**tool_args)

            # OBSERVE: Add tool call + result to conversation history
            print(f"  Observe: {tool_result}")
            print()

            # Add LLM tool request to history
            messages.append({
                "role": "assistant",
                "content": None,
                "tool_calls": [{
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_name,
                        "arguments": tool_call.function.arguments
                    }
                }]
            })

            # Add tool result to history
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

        else:
            # LLM gave a final answer — loop is done
            print("  Reason: LLM has enough information")
            print("  Act: Giving final answer")
            print()
            print("=" * 60)
            print("Final answer to customer:")
            print(message.content)
            return message.content

    # Safety: if we hit max_steps without final answer
    print("Max steps reached — escalating to human")
    return "I am unable to fully resolve your issue. Escalating to our support team."

# Test with our multi-step scenario
run_react_agent("My order ORD001 is very late, I want a refund please")

## Step 4: Test More Scenarios
Let us test two more cases:
1. A query that needs escalation
2. A simple query that needs only one step
This shows the loop is flexible — it uses as many steps as needed, no more.

In [ ]:
# Test 2: Simple one step query
print("\n" + "🔵 TEST 2: Simple query")
run_react_agent("What is the status of my order ORD003?")

print("\n" + "🔴 TEST 3: Problem agent cannot solve")
run_react_agent("My order ORD001 was delivered but the delivery partner was very rude to me")

## Summary

### What we built:
A Swiggy ReAct agent that loops through Reason → Act → Observe
until it has enough information to give a final answer.

### Key concepts:
- ReAct = Reason + Act. The loop every agent framework is built on.
- Conversation history grows with every step — this is the agent's memory.
- max_steps is a safety limit — without it a buggy agent loops forever.
- escalate_to_human is a fallback — agents need a safe exit.
- Tool descriptions drive behaviour — bad description = wrong decision.

### ReAct vs Session 1:
| Session 1 | Session 2 |
|---|---|
| One tool call | Multiple tool calls in a loop |
| Fixed steps | Dynamic — stops when ready |
| No memory between calls | Full conversation history |
| No fallback | escalate_to_human fallback |

### What frameworks like LangGraph do:
Everything we wrote manually today — the loop, history management,
step counting, tool dispatch — LangGraph does automatically.
Now you understand what is happening under the hood.

### AWS Equivalent:
| This session | AWS |
|---|---|
| ReAct loop | Amazon Bedrock Agents orchestration loop |
| max_steps | Bedrock Agent max iterations setting |
| escalate_to_human | Bedrock Agent human escalation action group |
| conversation history | Bedrock Agent session memory |

### Next session:
Module 5 Session 3 — Agents with LangGraph
We refactor this same loop using LangGraph nodes and edges
so you see exactly what the framework does for you.